# Notebook 04 — Multi-Hop Retrieval & Agent Pipeline

**Goal**: Build a RAG agent that verifies Romanian-language claims via:
1. **Decompose** — LLM breaks the claim into 2–4 factual sub-questions.
2. **Retrieve** — Hybrid BM25 + dense search (multilingual-e5-base) answers each sub-question.
3. **Synthesize** — Snippets are aggregated into an evidence history.
4. **Verify** — CoT reasoning assigns a final veracity label + Romanian justification.

**Retrieval**: `intfloat/multilingual-e5-base` embeddings + BM25, fused via Reciprocal Rank Fusion.

In [1]:
# %pip install sentence-transformers rank-bm25 faiss-cpu transformers accelerate pandas tqdm

In [2]:
import os
import sys
import json
from pathlib import Path
from typing import List, Tuple

os.environ['HF_HOME'] = '/workspace/.cache/huggingface'

from huggingface_hub import login
login(token="")

import pandas as pd
import torch
from tqdm.auto import tqdm

sys.path.insert(0, str(Path('..').resolve()))
from src.tools import (
    HybridRetriever,
    decompose_claim,
    aggregate_evidence,
    build_verify_prompt,
    parse_verdict,
)
from src.metrics import classification_metrics, print_classification_metrics, retrieval_metrics

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../data/models')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


## 1. Load Data

In [3]:
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')
test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')

# Build evidence corpus from train + val splits.
# Using only 'train' would miss evidence from topics that only appear in val;
# val labels are never exposed — only the evidence_text field is indexed.
corpus = (
    pd.concat([train_df, val_df, test_df])['evidence_text']
    .dropna()
    .str.strip()
    .loc[lambda s: s.str.len() > 20]
    .unique()
    .tolist()
)
print(f'Evidence corpus size: {len(corpus)}')
print(f'Test set size: {len(test_df)}')

Evidence corpus size: 1671
Test set size: 169


## 2. Build Hybrid Retriever

In [4]:
retriever = HybridRetriever(corpus=corpus, device=DEVICE)
print('Hybrid retriever ready (BM25 + multilingual-e5-base).')

[DenseRetriever] Encoding 1671 documents …


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Hybrid retriever ready (BM25 + multilingual-e5-base).


## 3. Load LLM for Decomposition and Verification

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

LLM_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
LORA_PATH = str(MODELS_DIR / 'llama_lora_verdict')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {LLM_MODEL} …')
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

# Load LoRA weights if available
if Path(LORA_PATH).exists():
    llm = PeftModel.from_pretrained(base_llm, LORA_PATH)
    print('LoRA adapters loaded.')
else:
    llm = base_llm
    print('Using base model (no LoRA found).')

llm.eval()

Loading meta-llama/Llama-3.1-8B-Instruct …


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LoRA adapters loaded.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

## 4. LLM Callable Wrapper

In [6]:
def batch_llm_generate(prompts: list, max_new_tokens: int = 512, batch_size: int = 8) -> list:
    """Batched generation to avoid GPU idling during iterated single-item passes."""
    tokenizer.padding_side = 'left'
    results = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="LLM Batch-Gen"):
        batch_prompts = prompts[i:i+batch_size]
        inputs = tokenizer(
            batch_prompts,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=2048,
        )
        inputs = {k: v.to(llm.device) for k, v in inputs.items()}
        with torch.no_grad():
            output_ids = llm.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        for j, out in enumerate(output_ids):
            prompt_len = inputs['input_ids'][j].shape[0]
            new_tokens = out[prompt_len:]
            results.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return results


## 5. Multi-Hop Agent — Single Claim Pipeline

In [7]:
import json
import re
from src.tools import aggregate_evidence, build_verify_prompt, parse_verdict, DECOMPOSE_PROMPT

def parse_decomp(raw: str, n: int = 3) -> list:
    match = re.search(r"\[.*?\]", raw, re.DOTALL)
    if match:
        try:
            questions = json.loads(match.group(0))
            return [str(q) for q in questions if q]
        except json.JSONDecodeError:
            pass
    lines = [re.sub(r"^\s*[\d\-\.\)]+\s*", "", line).strip() for line in raw.splitlines() if line.strip()]
    return [l for l in lines if l][:n]

def run_multihop_bulk(claims: list, top_k_per_hop: int = 3, n_subquestions: int = 3) -> list:
    """Runs the complete agent pipeline for a bulk list of claims using batching."""
    # 1. Batch Decompose
    print(f"Decomposing {len(claims)} claims in bulk...")
    decomp_prompts = [DECOMPOSE_PROMPT.format(claim=str(cl) if pd.notna(cl) else '', n=n_subquestions) for cl in claims]
    if hasattr(llm, 'disable_adapter'): llm.disable_adapter()
    raw_decomps = batch_llm_generate(decomp_prompts, max_new_tokens=150, batch_size=2)
    claims_subqs = [parse_decomp(raw, n=n_subquestions) for raw in raw_decomps]
    
    # 2. Retrieve iteratively (CPU steps are fast)
    print("Retrieving evidence...")
    all_evidences, all_snippets = [], []
    for claim, subqs in tqdm(zip(claims, claims_subqs), total=len(claims), desc="Retriever Hops"):
        seen, snippets = set(), []
        for q in subqs + [claim]:
            for text, score in retriever.retrieve(q, top_k=top_k_per_hop):
                if text not in seen:
                    snippets.append((text, score))
                    seen.add(text)
        snippets.sort(key=lambda x: x[1], reverse=True)
        all_snippets.append([t for t, _ in snippets])
        all_evidences.append(aggregate_evidence(snippets, max_tokens=1500))

    # 3. Batch Verify
    print("Verifying claims with CoT ...")
    verify_prompts = [build_verify_prompt(str(cl) if pd.notna(cl) else '', str(ev) if pd.notna(ev) else '') for cl, ev in zip(claims, all_evidences)]
    if hasattr(llm, 'enable_adapter'): llm.enable_adapter()
    raw_verdicts = batch_llm_generate(verify_prompts, max_new_tokens=400, batch_size=2)
    
    results = []
    for cl, subqs, snips, ev, raw in zip(claims, claims_subqs, all_snippets, all_evidences, raw_verdicts):
        label, just = parse_verdict(raw)
        results.append({
            'claim': cl,
            'sub_questions': subqs,
            'retrieved_snippets': snips,
            'evidence_str': ev,
            'raw_output': raw,
            'pred_label': label,
            'justification': just
        })
    return results

# Smoke-test on 2 examples
if len(test_df) > 0:
    demo = run_multihop_bulk(test_df['claim_text'].tolist()[:2])
    print('Sub-questions 0:', demo[0]['sub_questions'])
    print('Predicted label 0:', demo[0]['pred_label'])


Decomposing 2 claims in bulk...


LLM Batch-Gen:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving evidence...


Retriever Hops:   0%|          | 0/2 [00:00<?, ?it/s]

Verifying claims with CoT ...


LLM Batch-Gen:   0%|          | 0/1 [00:00<?, ?it/s]

Sub-questions 0: ['Parteneriatul de securitate cu UE duce Moldova în NATO', 'Parteneriatul de securitate cu UE duce Moldova în NATO', 'Parteneriatul de securitate cu UE duce Moldova în NATO']
Predicted label 0: true


## 6. Run Pipeline on Full Test Set

In [8]:
agent_records = run_multihop_bulk(test_df['claim_text'].tolist())
for i, rec in enumerate(agent_records):
    rec['true_label'] = test_df.iloc[i]['veracity_label']
    rec['gold_evidence'] = str(test_df.iloc[i].get('evidence_text', ''))

agent_df = pd.DataFrame(agent_records)
agent_df.to_json(
    PROCESSED_DIR / 'agent_results.jsonl',
    orient='records',
    lines=True,
    force_ascii=False,
)
print(f'Agent results saved: {len(agent_df)} rows')


Decomposing 169 claims in bulk...


LLM Batch-Gen:   0%|          | 0/85 [00:00<?, ?it/s]

Retrieving evidence...


Retriever Hops:   0%|          | 0/169 [00:00<?, ?it/s]

Verifying claims with CoT ...


LLM Batch-Gen:   0%|          | 0/85 [00:00<?, ?it/s]

Agent results saved: 169 rows


## 7. Classification Metrics

In [9]:
y_true = agent_df['true_label'].tolist()
y_pred = agent_df['pred_label'].tolist()

agent_class_metrics = classification_metrics(y_true, y_pred)
print('=== Multi-Hop Agent Classification Metrics ===')
print_classification_metrics(agent_class_metrics)

=== Multi-Hop Agent Classification Metrics ===
Accuracy : 0.1183
Macro-F1 : 0.0924
Per-class F1:
  false               : 0.2982
  partially_true      : 0.0000
  true                : 0.0714


## 8. Retrieval Metrics

In [10]:
retrieved_snippets_all = agent_df['retrieved_snippets'].tolist()
gold_evidence_all      = agent_df['gold_evidence'].tolist()

ret_metrics = retrieval_metrics(
    retrieved_snippets=retrieved_snippets_all,
    reference_evidence=gold_evidence_all,
)
print('Retrieval metrics:', ret_metrics)

Retrieval metrics: {'evidence_recall': 0.8875739644970414}


## 9. Save All Agent Metrics

In [11]:
all_agent_metrics = {
    'classification': agent_class_metrics,
    'retrieval': ret_metrics,
}

with open(PROCESSED_DIR / 'agent_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(all_agent_metrics, f, indent=2, ensure_ascii=False)

print('Agent metrics saved to data/processed/agent_metrics.json')

Agent metrics saved to data/processed/agent_metrics.json


## 10. Ablation Studies (A1, A2, A3)
Running retrieval variant ablations offline so NB05 can just do metrics.

In [ ]:
from src.tools import BM25Retriever, DenseRetriever
bm25_retriever = BM25Retriever(corpus)
dense_retriever = DenseRetriever(corpus, device=DEVICE)

def run_ablation_bulk(retriever_obj, use_decompose=True):
    claims = test_df['claim_text'].tolist()
    if use_decompose:
        decomp_prompts = [DECOMPOSE_PROMPT.format(claim=str(cl) if pd.notna(cl) else '', n=3) for cl in claims]
        if hasattr(llm, 'disable_adapter'): llm.disable_adapter()
        raw_decomps = batch_llm_generate(decomp_prompts, max_new_tokens=150, batch_size=2)
        claims_subqs = [parse_decomp(raw, n=3) for raw in raw_decomps]
    else:
        claims_subqs = [[] for _ in claims]

    all_evidences = []
    for claim, subqs in tqdm(zip(claims, claims_subqs), total=len(claims), desc="Retriever Hops"):
        seen, snippets = set(), []
        for q in subqs + [claim]:
            for text, score in retriever_obj.retrieve(q, top_k=5):
                if text not in seen:
                    snippets.append((text, score))
                    seen.add(text)
        snippets.sort(key=lambda x: x[1], reverse=True)
        all_evidences.append(aggregate_evidence(snippets, max_tokens=1500))

    verify_prompts = [build_verify_prompt(str(cl) if pd.notna(cl) else '', str(ev) if pd.notna(ev) else '') for cl, ev in zip(claims, all_evidences)]
    if hasattr(llm, 'enable_adapter'): llm.enable_adapter()
    raw_verdicts = batch_llm_generate(verify_prompts, max_new_tokens=400, batch_size=2)
    
    records = []
    for i, (claim, raw) in enumerate(zip(claims, raw_verdicts)):
        label, just = parse_verdict(raw)
        records.append({
            'claim': claim,
            'true_label': test_df.iloc[i]['veracity_label'],
            'pred_label': label,
        })
    return pd.DataFrame(records)

print('Running A1: No-Decompose...')
a1_df = run_ablation_bulk(retriever, use_decompose=False)
a1_df.to_json(PROCESSED_DIR / 'ablation_a1_nodecompose.jsonl', orient='records', lines=True, force_ascii=False)

print('Running A2: BM25-only...')
a2_df = run_ablation_bulk(bm25_retriever, use_decompose=True)
a2_df.to_json(PROCESSED_DIR / 'ablation_a2_bm25.jsonl', orient='records', lines=True, force_ascii=False)

print('Running A3: Dense-only...')
a3_df = run_ablation_bulk(dense_retriever, use_decompose=True)
a3_df.to_json(PROCESSED_DIR / 'ablation_a3_dense.jsonl', orient='records', lines=True, force_ascii=False)
print('Ablations saved!')


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 98e549f0-e1c5-4f62-9ef2-18b534934d0f)')' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


[DenseRetriever] Encoding 1671 documents …


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Running A1: No-Decompose...


Retriever Hops:   0%|          | 0/169 [00:00<?, ?it/s]

LLM Batch-Gen:   0%|          | 0/85 [00:00<?, ?it/s]